In [1]:
# Install (only needed first time)
!pip install -q sentence-transformers faiss-cpu openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 39.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 24.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 54.3 MB/s eta 0:00:0000:0100:01


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# ===============================
# 1️⃣ Load Excel (All Sheets)
# ===============================

file_path = "/kaggle/input/product-data/All Products Data.xlsx"

all_sheets = pd.read_excel(file_path, sheet_name=None)
df = pd.concat(all_sheets.values(), ignore_index=True)

print("Total Products (Raw):", len(df))

# ===============================
# 2️⃣ Clean & Normalize
# ===============================

df = df.fillna("")

df.columns = df.columns.astype(str).str.strip().str.lower()
df = df.loc[:, ~df.columns.duplicated()]
df = df.applymap(lambda x: str(x).strip().lower())

# Ensure product type exists
if "product type" not in df.columns:
    raise ValueError("Column 'product type' not found in Excel.")

# Remove empty categories BEFORE indexing
df["product type"] = df["product type"].replace("", np.nan)
df = df.dropna(subset=["product type"])

# 🔥 CRITICAL: Reset index BEFORE embedding
df = df.reset_index(drop=True)

print("Total Products (After Cleaning):", len(df))

# ===============================
# 3️⃣ Build Embedding Text
# ===============================

def build_embedding_text(row):
    text = "This is a product listing.\n\n"
    for col in df.columns:
        if row[col] != "":
            text += f"{col}: {row[col]}\n"
    return text.strip()

corpus = df.apply(build_embedding_text, axis=1).tolist()

# ===============================
# 4️⃣ Load Embedding Model
# ===============================

model = SentenceTransformer("BAAI/bge-base-en-v1.5")

# ===============================
# 5️⃣ Create Product Embeddings
# ===============================

embeddings = model.encode(
    corpus,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index created with", index.ntotal, "products")

# ===============================
# 6️⃣ Improved Semantic Category Detection
# ===============================

categories = df["product type"].unique().tolist()

category_texts = []

for cat in categories:
    sample_products = df[df["product type"] == cat]["product name"].head(10).tolist()
    combined_text = f"Category: {cat}. Example products: " + ", ".join(sample_products)
    category_texts.append(combined_text)

category_embeddings = model.encode(
    category_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

def detect_category_semantic(query):
    query_vector = model.encode(
        "Represent this product category search query: " + query.lower(),
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarities = np.dot(category_embeddings, query_vector)
    best_index = np.argmax(similarities)

    return categories[best_index]

# ===============================
# 7️⃣ Query Embedding
# ===============================

def embed_query(query):
    return model.encode(
        "Represent this product search query for retrieval: " + query.lower(),
        convert_to_numpy=True,
        normalize_embeddings=True
    )

# ===============================
# 8️⃣ Final Search Function
# ===============================

def search_products(query, top_k=5):

    detected_category = detect_category_semantic(query)

    # Strict filtering
    filtered_indices = df.index[df["product type"] == detected_category].tolist()

    if len(filtered_indices) == 0:
        print("No products found in detected category.")
        return pd.DataFrame()

    query_vector = embed_query(query)

    scores, indices = index.search(
        np.array([query_vector]),
        50
    )

    SIMILARITY_THRESHOLD = 0.62  # adjust if needed

    valid_results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx in filtered_indices and score >= SIMILARITY_THRESHOLD:
            valid_results.append((idx, score))

    if len(valid_results) == 0:
        print("No relevant products found.")
        print("Detected Category:", detected_category)
        return pd.DataFrame()

    valid_results = sorted(valid_results, key=lambda x: x[1], reverse=True)[:top_k]

    result_indices = [x[0] for x in valid_results]
    result_scores = [x[1] for x in valid_results]

    results = df.iloc[result_indices].copy()
    results["similarity_score"] = result_scores

    print("Detected Category:", detected_category)

    return results


2026-02-14 18:42:00.518210: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771094520.684598      38 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771094520.735275      38 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Total Products (Raw): 1769
Total Products (After Cleaning): 1614


/tmp/ipykernel_38/3081248931.py:25: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: str(x).strip().lower())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/51 [00:00<?, ?it/s]

FAISS index created with 1614 products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [3]:
# ===============================
# 🔎 TEST
# ===============================

query = "fitness tent"
results = search_products(query, top_k=5)
results.head()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Detected Category: fitness and outdoor


,product type,product code,product name,brand,model,color,description,price,quantity in stock,specifications,...,special features,subcategory,material / fabric,return policy,occasion,closure type,toe shape,return policy / expiry date,dimensions / size,similarity_score
859,fitness and outdoor,newce001,newdru waterproof camping tent for 3-4 person ...,newdru,,multi-color,premium camping equipment from newdru. high-qu...,,,,...,"waterproof, lightweight",,,,,,,,,0.667779
862,fitness and outdoor,trece001,arura (label) portable tent for camping | 5-10...,trek,,multi-color,premium camping equipment from trek. high-qual...,,,,...,portable,,,,,,,,,0.667756
865,fitness and outdoor,pyuce001,pyu 2025 6 person waterproof camping tent | fa...,pyu,,multi-color,premium camping equipment from pyu. high-quali...,,,,...,waterproof,,,,,,,,,0.663574
861,fitness and outdoor,indce001,inditradition polyester waterproof camping & p...,inditradition,,multi-color,premium camping equipment from inditradition. ...,,,,...,waterproof,,,,,,,,,0.659945
868,fitness and outdoor,rooce001,roots & leaf waterproof gazebo tent/canopy 10 ...,roots,,blue,premium camping equipment from roots. high-qua...,,,,...,waterproof,,,,,,,,,0.659049


In [4]:
def print_categories(df):
    if "product type" not in df.columns:
        print("Column 'product type' not found.")
        return
    
    categories = (
        df["product type"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )
    
    print("Total Categories:", len(categories))
    print("\nCategory List:")
    
    for i, cat in enumerate(sorted(categories), 1):
        print(f"{i}. {cat}")

# Call function
print_categories(df)


Total Categories: 13

Category List:
1. accessories
2. clothing
3. fitness and outdoor
4. footwear
5. fragrance
6. haircare
7. laptop
8. makeup
9. phone
10. skincare
11. sports
12. tablet
13. television


# **Intent Classification**

In [5]:
%%bash
cat > intents.json <<'JSON'
{
  "intents": [

    {
      "tag": "profile_changes",
      "patterns": [
        "Update my profile",
        "Change my address",
        "Edit profile information",
        "Modify my account details",
        "Change phone number",
        "Update billing address",
        "Edit my name",
        "I want to change my profile",
        "I want to update my personal info",
        "I want to update my details",
        "Change my information",
        "Update my email",
        "Revise my profile",
        "Adjust my phone number",
        "Correct my name",
        "Modify profile settings"
      ]
    },
    {
      "tag": "browse_product",
      "patterns": [
    
        "show me products",
        "show me items",
        "show me what you have",
        "what are you selling",
        "what can i buy",
        "browse items",
        "browse store",
        "browse catalog",
        "open catalog",
        "view products",
        "list items",
        "display items",
        "explore store",
        "explore products",
        "see what's available",
        "see available items",
    
        "i want to buy something",
        "i want to purchase something",
        "i want to get something",
        "i need something",
        "i need a product",
        "i want something new",
        "i am looking to buy",
        "i plan to buy something",
        "help me buy something",
    
        "find me something",
        "find a product",
        "search for something",
        "search product",
        "look for something",
        "find items",
        "help me find",
    
        "show electronics",
        "show clothing",
        "show fashion",
        "show groceries",
        "show shoes",
        "show mobiles",
        "show laptops",
        "browse electronics",
        "browse fashion",
        "browse groceries",
        "show me phones",
        "show me laptops",
        "show me shoes",
    
        "show me leather shoes",
        "show leather boots",
        "give me boots",
        "find iphone",
        "find samsung phone",
        "show iphone",
        "show nike shoes",
        "show adidas sneakers",
        "i need sneakers",
        "i need boots",
        "i need headphones",
        "i need a laptop",
        "i need a phone",
        "i need a tv",
        "i need a jacket",
        "i need a watch",
    
        "show apple products",
        "show samsung phones",
        "show dell laptops",
        "show sony headphones",
    
        "what do you have",
        "anything good to buy",
        "show me some stuff",
        "show me something cool",
        "what's trending",
        "what's popular",
        "show new arrivals",
    
        "show cheap phones",
        "show affordable laptops",
        "products under 5000",
        "phones below 20000",
    
        "best phone available",
        "best laptop to buy",
        "top rated shoes",
        "top selling products",
    
        "get me a phone",
        "get me shoes",
        "bring up laptops",
        "display jackets"
      ]
    },

    {
      "tag": "add_to_cart",
      "patterns": [
        "Add this to my cart",
        "Put this in my cart",
        "Add item to basket",
        "Add headphones to cart",
        "Please add this to my cart",
        "Put that in my basket",
        "Add this product to my basket",
        "Keep this in my cart",
        "Place it in my shopping bag",
        "Add all to cart",
        "Include this in my cart"
      ]
    },

    {
      "tag": "remove_from_cart",
      "patterns": [
        "Remove this from cart",
        "Delete this from my cart",
        "Take this out of my cart",
        "Remove item",
        "Remove headphones from cart",
        "Delete that from my basket",
        "Remove it",
        "I don't want this in my cart",
        "Take that out of my basket",
        "Clear this from my cart",
        "Cancel this item from cart",
        "Discard this from shopping bag"
      ]
    },

    {
      "tag": "cart_detail",
      "patterns": [
        "What's in my cart",
        "Show my cart",
        "View cart",
        "What's in my shopping bag",
        "Cart contents",
        "Read my cart items",
        "How many items in my cart",
        "Open my cart",
        "Show me what I've added",
        "Check my cart items",
        "Review cart"
      ]
    },

    {
      "tag": "place_order",
      "patterns": [
        "Checkout now",
        "Place my order",
        "Confirm purchase",
        "Proceed to payment",
        "Order this item",
        "Finalize purchase",
        "Confirm order now",
        "Go ahead with payment",
        "Pay for items",
        "Complete my order"
      ]
    },

    {
      "tag": "cancel_order",
      "patterns": [
        "Cancel my order",
        "Please cancel my purchase",
        "Stop my order",
        "Abort my order",
        "Undo my order",
        "Delete my order",
        "Discard my order"
      ]
    },

    {
      "tag": "track_order",
      "patterns": [
        "Track my order",
        "Where is my package",
        "Order status",
        "Track shipment",
        "Where is my delivery",
        "Track my delivery",
        "When will my delivery arrive",
        "Is my package on the way"
      ]
    },

    {
      "tag": "order_history",
      "patterns": [
        "Show my past orders",
        "Order history",
        "What orders have I placed",
        "List my previous purchases",
        "Show past purchases",
        "Display purchase history",
        "See previous orders"
      ]
    },

    {
      "tag": "logout",
      "patterns": [
        "Log me out",
        "Sign me out",
        "Logout",
        "End session",
        "Exit account",
        "Sign off",
        "Close my session"
      ]
    }

  ]
}
JSON
echo "wrote intents.json"

wrote intents.json


In [ ]:
pip install sentence-transformers scikit-learn numpy

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [7]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ==============================
# 1️⃣ Load Intents
# ==============================

with open("intents.json", "r", encoding="utf-8") as f:
    data = json.load(f)

pattern_texts = []
labels = []

for intent in data["intents"]:
    tag = intent["tag"]
    for pattern in intent["patterns"]:
        pattern_texts.append(pattern)
        labels.append(tag)

# ==============================
# 2️⃣ Load MiniLM Model
# ==============================

model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode all patterns once (important for speed)
pattern_embeddings = model.encode(pattern_texts)

# ==============================
# 3️⃣ Intent Prediction Function
# ==============================

def predict_intent(query, threshold=0.38):
    query_embedding = model.encode([query])
    
    similarities = cosine_similarity(query_embedding, pattern_embeddings)
    
    best_index = np.argmax(similarities)
    best_score = similarities[0][best_index]
    predicted_intent = labels[best_index]
    
    if best_score < threshold:
        return "fallback", float(best_score)
    
    return predicted_intent, float(best_score)

# ==============================
# 4️⃣ Run Interactive Loop
# ==============================

print("\n🚀 Intent Classifier Ready (type 'exit' to quit)\n")

while True:
    user_query = input("User: ")
    
    if user_query.lower() == "exit":
        break
    
    intent, confidence = predict_intent(user_query)
    
    print(f"Predicted Intent: {intent}")
    print(f"Confidence: {confidence:.2f}")
    print("-" * 40)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]


🚀 Intent Classifier Ready (type 'exit' to quit)



User:  exit
